# Comparing LSTM and Transformer Models for Oil Temperature Forecasting

Ayaa Asoba and Xavier Bruneau


## Data Preprocessing

In [ ]:
# Import necessary libraries
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import random

# Set random seeds for reproducibility
np.random.seed(13)
tf.random.set_seed(13)
random.seed(13)

In [ ]:
# Load the dataset
path_data = os.getcwd()
df = pd.read_csv(os.path.join(path_data, "ETTh1.csv"))

# Set 'date' column as index and convert to datetime, enforce hourly frequency
df.index = df['date']
df.index = pd.to_datetime(df.index)
df=df.drop(columns=['date'])
df = df.asfreq('h')

# Define target variable and feature columns
y = 'OT'
features = [col for col in df.columns if col != y]

## Exploratory Data Analysis

In [ ]:
# Plot oil temperature daily average over the entire period
df_daily = df.groupby(df.index.date).agg({y: 'mean'}).reset_index()
df_daily.columns = ['date', y]
df_daily['date'] = pd.to_datetime(df_daily['date'])

plt.figure(figsize=(14, 5))
plt.plot(df_daily['date'], df_daily[y], linewidth=2, color='steelblue', label='Daily Average OT')
plt.xlabel('Date', fontsize=11)
plt.ylabel('Oil Temperature (°C)', fontsize=11)
plt.title('Oil Temperature Series over the 2016-2018 Period', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot all predictor variables and target variable daily average over the entire period

df_daily_multi = df.groupby(df.index.date).agg({col: 'mean' for col in features + [y]}).reset_index()
df_daily_multi.columns = ['date'] + features + [y]
df_daily_multi['date'] = pd.to_datetime(df_daily_multi['date'])

colors = ['#2563EB', '#0EA5E9', '#6366F1', '#8B5CF6', '#06B6D4', '#3B82F6', '#E74C3C']
plt.figure(figsize=(14, 6))
for idx, col in enumerate(features + [y]):
    plt.plot(df_daily_multi['date'], df_daily_multi[col], label=col, linewidth=1.5, color=colors[idx % len(colors)])
plt.xlabel('Date', fontsize=11)
plt.ylabel('Value', fontsize=11)
plt.title('Time Series of All Predictors and Target (Daily Averages)', fontsize=13, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Time Series Decomposition using STL
from statsmodels.tsa.seasonal import STL

decomposition = STL(df[y], period=24, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10))

# Observed
axes[0].plot(df.index, decomposition.observed, linewidth=1.5, color='steelblue')
axes[0].set_ylabel('Observed', fontsize=10, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Trend
axes[1].plot(df.index, decomposition.trend, linewidth=1.5, color='orange')
axes[1].set_ylabel('Trend', fontsize=10, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Seasonal
axes[2].plot(df.index, decomposition.seasonal, linewidth=1.5, color='green')
axes[2].set_ylabel('Seasonal', fontsize=10, fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Residual
axes[3].plot(df.index, decomposition.resid, linewidth=1, color='red', alpha=0.7)
axes[3].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[3].set_ylabel('Residual', fontsize=10, fontweight='bold')
axes[3].set_xlabel('Date', fontsize=10)
axes[3].grid(True, alpha=0.3)

plt.suptitle('Time Series Decomposition (STL, period=24h)', fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## Modeling

### Model A: LSTM (Long Short-Term Memory) Model for Time Series Forecasting

In [ ]:
# Import additional libraries for modeling and evaluation
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
# Prepare data for both models
lookback = 168  # 1 week = 168 hours
forecast_horizon = 24  # predict next 24 hours

train_split_idx = int(len(df) * 0.8)
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(df[[y]].iloc[:train_split_idx])
scaled_data = scaler.transform(df[[y]])

# Create sequences
from functions import create_sequences
X, y_lstm = create_sequences(scaled_data, lookback, forecast_horizon)

# 80-20 chrono split (no shuffle — time series)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y_lstm[:train_size], y_lstm[train_size:]

# Reshape to [samples, timesteps, features]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test  = X_test.reshape((X_test.shape[0],  X_test.shape[1],  1))

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")

In [ ]:
# Build LSTM Model
from functions import build_lstm_model
model = build_lstm_model(lookback, forecast_horizon, learning_rate=0.001)
model.summary()

In [ ]:
# LSTM Training with callbacks
import time

# Training hyperparameters (shared across both models)
BATCH_SIZE = 32
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 6
LR_REDUCE_PATIENCE = 4
LR_REDUCE_FACTOR = 0.5
MIN_LR = 1e-5

lstm_callbacks = [
    EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=LR_REDUCE_FACTOR, patience=LR_REDUCE_PATIENCE, min_lr=MIN_LR, verbose=1),
]
start_time = time.time()
history = model.fit(
    X_train, y_train,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=lstm_callbacks,
    verbose=0,
)
end_time = time.time()
lstm_time = end_time - start_time
print(f"LSTM stopped at epoch {len(history.history['loss'])} / {MAX_EPOCHS}")
print(f"Training time: {lstm_time:.2f} seconds")

In [ ]:
# Plot LSTM training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2, color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2, color='coral')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss (MSE)', fontsize=11)
axes[0].set_title('LSTM Training Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2, color='steelblue')
axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2, color='coral')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_title('LSTM Training MAE', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Make predictions
lstm_train_pred = model.predict(X_train)  # (samples, 24)
lstm_test_pred = model.predict(X_test)    # (samples, 24)

# Inverse transform to get original scale
# Reshape predictions to (total_steps, 1) for scaler
lstm_train_pred_original = scaler.inverse_transform(lstm_train_pred.reshape(-1, 1)).reshape(lstm_train_pred.shape)
lstm_test_pred_original = scaler.inverse_transform(lstm_test_pred.reshape(-1, 1)).reshape(lstm_test_pred.shape)
lstm_train_original = scaler.inverse_transform(y_train.reshape(-1, 1)).reshape(y_train.shape)
lstm_test_original = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)

lstm_train_rmse = np.sqrt(mean_squared_error(lstm_train_original, lstm_train_pred_original))
lstm_test_rmse = np.sqrt(mean_squared_error(lstm_test_original, lstm_test_pred_original))
lstm_train_mae = mean_absolute_error(lstm_train_original, lstm_train_pred_original)
lstm_test_mae = mean_absolute_error(lstm_test_original, lstm_test_pred_original)
lstm_test_r2 = r2_score(lstm_test_original.flatten(), lstm_test_pred_original.flatten())

print(f"\nLSTM Model Performance (168h → 24-step):")
print(f"Train RMSE: {lstm_train_rmse:.4f}")
print(f"Test  RMSE: {lstm_test_rmse:.4f}")
print(f"Train MAE : {lstm_train_mae:.4f}")
print(f"Test  MAE : {lstm_test_mae:.4f}")
print(f"Test  R²  : {lstm_test_r2:.4f}")
print(f"Training time: {lstm_time:.2f} seconds")

### Model B: Transformers

In [ ]:
# Transformer model preparation
# Import additional libraries for Transformer model
from sklearn.preprocessing import StandardScaler
from functions import make_windows, build_transformer_model

# Specify input and forecast lengths for the Transformer model
INPUT_LEN    = 168   # 1 week
FORECAST_LEN = 24    # 24 hours

# Extract trend using STL and use it as an additional feature for the Transformer model
series = df[y].values
feature_matrix = df[features].values
train_end = int(len(series) * 0.8)

trend = STL(series, period=24, robust=True).fit().trend

# Scale features and trend separately
scaler_x = StandardScaler().fit(feature_matrix[:train_end])
features_scaled = scaler_x.transform(feature_matrix)        # (N, 6)

scaler_trend = StandardScaler().fit(trend[:train_end].reshape(-1, 1))
trend_scaled = scaler_trend.transform(trend.reshape(-1, 1))    # (N, 1)
features_scaled = np.hstack([features_scaled, trend_scaled]) # (N, 7)

scaler_y = StandardScaler().fit(series[:train_end].reshape(-1, 1))
targets_scaled = scaler_y.transform(series.reshape(-1, 1)).flatten()

# Create features array: [series, trend]
raw_scaled = scaler_y.transform(series.reshape(-1, 1)).flatten()
transformer_features = np.hstack([raw_scaled.reshape(-1, 1), trend_scaled])  # (N, 2)

# Create windows for the Transformer model
trans_X, trans_y = make_windows(transformer_features, targets_scaled, INPUT_LEN, FORECAST_LEN)

# 80-20 chronological split (no shuffle — time series)
trans_train_size = int(len(trans_X) * 0.8)
trans_X_train, trans_X_test = trans_X[:trans_train_size], trans_X[trans_train_size:]
trans_y_train, trans_y_test = trans_y[:trans_train_size], trans_y[trans_train_size:]

print(f"Transformer training set : {trans_X_train.shape}")
print(f"Transformer test set     : {trans_X_test.shape}")

# Build Transformer Model
transformer_model = build_transformer_model(INPUT_LEN, FORECAST_LEN, learning_rate=0.001, num_features=2)
transformer_model.summary()

In [ ]:
# Train Transformer with callbacks (using same hyperparameters as LSTM)
trans_callbacks = [
    EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=LR_REDUCE_FACTOR, patience=LR_REDUCE_PATIENCE, min_lr=MIN_LR, verbose=1),
]
start_time = time.time()
trans_history = transformer_model.fit(
    trans_X_train, trans_y_train,
    validation_split=0.2,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=trans_callbacks,
    verbose=0,
)
end_time = time.time()
transformer_time = end_time - start_time
print(f"Transformer stopped at epoch {len(trans_history.history['loss'])} / {MAX_EPOCHS}")
print(f"Transformer training time: {transformer_time:.2f} seconds")
print(f"\nTraining Hyperparameters (both models):")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Early stopping patience: {EARLY_STOP_PATIENCE}")
print(f"  LR reduce patience: {LR_REDUCE_PATIENCE}")
print(f"  LR reduce factor: {LR_REDUCE_FACTOR}")
print(f"  Weight decay (L2): 1e-6")

In [ ]:
# Plot Transformer training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(trans_history.history['loss'], label='Train Loss', linewidth=2, color='steelblue')
axes[0].plot(trans_history.history['val_loss'], label='Val Loss', linewidth=2, color='coral')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss (MSE)', fontsize=11)
axes[0].set_title('Transformer Training Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(trans_history.history['mae'], label='Train MAE', linewidth=2, color='steelblue')
axes[1].plot(trans_history.history['val_mae'], label='Val MAE', linewidth=2, color='coral')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_title('Transformer Training MAE', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Make predictions and inverse transform
trans_train_pred_scaled = transformer_model.predict(trans_X_train)
trans_test_pred_scaled  = transformer_model.predict(trans_X_test)

trans_train_pred = scaler_y.inverse_transform(trans_train_pred_scaled.reshape(-1, 1)).reshape(trans_train_pred_scaled.shape)
trans_test_pred  = scaler_y.inverse_transform(trans_test_pred_scaled.reshape(-1, 1)).reshape(trans_test_pred_scaled.shape)
trans_train_true = scaler_y.inverse_transform(trans_y_train.reshape(-1, 1)).reshape(trans_y_train.shape)
trans_test_true  = scaler_y.inverse_transform(trans_y_test.reshape(-1, 1)).reshape(trans_y_test.shape)

# Compute metrics
trans_train_rmse = np.sqrt(mean_squared_error(trans_train_true, trans_train_pred))
trans_test_rmse  = np.sqrt(mean_squared_error(trans_test_true, trans_test_pred))
trans_train_mae  = mean_absolute_error(trans_train_true, trans_train_pred)
trans_test_mae   = mean_absolute_error(trans_test_true, trans_test_pred)
trans_test_r2    = r2_score(trans_test_true.flatten(), trans_test_pred.flatten())

print(f"\nTransformer Model Performance (168h → 24-step with STL trend):") 
print(f"Train RMSE: {trans_train_rmse:.4f}")
print(f"Test  RMSE: {trans_test_rmse:.4f}")
print(f"Train MAE : {trans_train_mae:.4f}")
print(f"Test  MAE : {trans_test_mae:.4f}")
print(f"Test  R²  : {trans_test_r2:.4f}")
print(f"Training time: {transformer_time:.2f} seconds")

### Learning Rate Sensitivity Experiment

In [ ]:
# LSTM Learning Rate Experiment: Train 2 new rates, compare with original (0.001)
lstm_new_lrs = [0.0005, 0.005] 
lstm_lr_results = {'lr': [0.001], 'test_rmse': [lstm_test_rmse], 'test_mae': [lstm_test_mae], 'training_time': [lstm_time]}  # Start with original

for lr in lstm_new_lrs:
    # Build and train with different LR
    lstm_lr_model = build_lstm_model(lookback, forecast_horizon, learning_rate=lr)
    start_time = time.time()
    lstm_lr_callbacks = [
        EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=LR_REDUCE_FACTOR, patience=LR_REDUCE_PATIENCE, min_lr=MIN_LR, verbose=0),
    ]
    
    lstm_lr_history = lstm_lr_model.fit(
        X_train, y_train,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=lstm_lr_callbacks,
        verbose=0,
    )
    end_time = time.time()
    lstm_lr_time = end_time - start_time
    lstm_lr_pred = lstm_lr_model.predict(X_test, verbose=0)
    lstm_lr_pred_orig = scaler.inverse_transform(lstm_lr_pred.reshape(-1, 1)).reshape(lstm_lr_pred.shape)
    lstm_lr_test_orig = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    
    test_rmse = np.sqrt(mean_squared_error(lstm_lr_test_orig, lstm_lr_pred_orig))
    test_mae = mean_absolute_error(lstm_lr_test_orig, lstm_lr_pred_orig)
    
    lstm_lr_results['lr'].append(lr)
    lstm_lr_results['test_rmse'].append(test_rmse)
    lstm_lr_results['test_mae'].append(test_mae)
    lstm_lr_results['training_time'].append(lstm_lr_time)
    

In [ ]:
# Transformer Learning Rate Experiment: Train 2 new rates, compare with original (0.001)
transformer_new_lrs = [0.0005, 0.005]  
transformer_lr_results = {'lr': [0.001], 'test_rmse': [trans_test_rmse], 'test_mae': [trans_test_mae], 'training_time': [transformer_time]}  # Start with original

for lr in transformer_new_lrs:
    # Build and train with different LR using factory function
    transformer_lr_model = build_transformer_model(INPUT_LEN, FORECAST_LEN, learning_rate=lr, num_features=2)
    
    trans_lr_callbacks = [
        EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=LR_REDUCE_FACTOR, patience=LR_REDUCE_PATIENCE, min_lr=MIN_LR, verbose=0),
    ]
    start_time = time.time()
    transformer_lr_history = transformer_lr_model.fit(
        trans_X_train, trans_y_train,
        validation_split=0.2,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=trans_lr_callbacks,
        verbose=0,
    )
    end_time = time.time()
    trans_lr_time = end_time - start_time
    trans_lr_pred = transformer_lr_model.predict(trans_X_test, verbose=0)
    trans_lr_pred_orig = scaler_y.inverse_transform(trans_lr_pred.reshape(-1, 1)).reshape(trans_lr_pred.shape)
    trans_lr_test_orig = scaler_y.inverse_transform(trans_y_test.reshape(-1, 1)).reshape(trans_y_test.shape)
    
    test_rmse = np.sqrt(mean_squared_error(trans_lr_test_orig, trans_lr_pred_orig))
    test_mae = mean_absolute_error(trans_lr_test_orig, trans_lr_pred_orig)
    
    transformer_lr_results['lr'].append(lr)
    transformer_lr_results['test_rmse'].append(test_rmse)
    transformer_lr_results['test_mae'].append(test_mae)
    transformer_lr_results['training_time'].append(trans_lr_time)


In [ ]:
# Comparison: LSTM vs Transformer Learning Rate Sensitivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE comparison
axes[0].plot(lstm_lr_results['lr'], lstm_lr_results['test_rmse'], 'o-', linewidth=2.5, markersize=10, color='#2980b9', label='LSTM')
axes[0].plot(transformer_lr_results['lr'], transformer_lr_results['test_rmse'], 's-', linewidth=2.5, markersize=10, color='#e74c3c', label='Transformer')
lstm_orig_idx = lstm_lr_results['lr'].index(0.001)
trans_orig_idx = transformer_lr_results['lr'].index(0.001)
axes[0].plot(lstm_lr_results['lr'][lstm_orig_idx], lstm_lr_results['test_rmse'][lstm_orig_idx], 'o', markersize=12, 
             markerfacecolor='none', markeredgewidth=2, markeredgecolor='green')
axes[0].plot(transformer_lr_results['lr'][trans_orig_idx], transformer_lr_results['test_rmse'][trans_orig_idx], 's', markersize=12, 
             markerfacecolor='none', markeredgewidth=2, markeredgecolor='green')
axes[0].set_xscale('log')
axes[0].set_xlabel('Learning Rate (log scale)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Test RMSE (°C)', fontsize=11, fontweight='bold')
axes[0].set_title('Test RMSE vs Learning Rate', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# MAE comparison
axes[1].plot(lstm_lr_results['lr'], lstm_lr_results['test_mae'], 'o-', linewidth=2.5, markersize=10, color='#2980b9', label='LSTM')
axes[1].plot(transformer_lr_results['lr'], transformer_lr_results['test_mae'], 's-', linewidth=2.5, markersize=10, color='#e74c3c', label='Transformer')
axes[1].plot(lstm_lr_results['lr'][lstm_orig_idx], lstm_lr_results['test_mae'][lstm_orig_idx], 'o', markersize=12, 
             markerfacecolor='none', markeredgewidth=2, markeredgecolor='green')
axes[1].plot(transformer_lr_results['lr'][trans_orig_idx], transformer_lr_results['test_mae'][trans_orig_idx], 's', markersize=12, 
             markerfacecolor='none', markeredgewidth=2, markeredgecolor='green')
axes[1].set_xscale('log')
axes[1].set_xlabel('Learning Rate (log scale)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Test MAE (°C)', fontsize=11, fontweight='bold')
axes[1].set_title('Test MAE vs Learning Rate', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Learning Rate Sensitivity Comparison (Original 0.001 Highlighted)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Analysis
lstm_rmse_range = max(lstm_lr_results['test_rmse']) - min(lstm_lr_results['test_rmse'])
trans_rmse_range = max(transformer_lr_results['test_rmse']) - min(transformer_lr_results['test_rmse'])

print(f"\n RMSE Sensitivity (range across 3 learning rates):")
print(f"   LSTM:        {lstm_rmse_range:.4f}°C")
print(f"   Transformer: {trans_rmse_range:.4f}°C")
if lstm_rmse_range > trans_rmse_range:
    print(f"   → LSTM is {(lstm_rmse_range/trans_rmse_range):.2f}x more sensitive to learning rate")
else:
    print(f"   → Transformer is {(trans_rmse_range/lstm_rmse_range):.2f}x more sensitive to learning rate")

print(f"\nBest Learning Rates:")
lstm_best_idx = np.argmin(lstm_lr_results['test_rmse'])
trans_best_idx = np.argmin(transformer_lr_results['test_rmse'])
print(f"   LSTM:        LR={lstm_lr_results['lr'][lstm_best_idx]:.4f} → RMSE={lstm_lr_results['test_rmse'][lstm_best_idx]:.4f}°C")
print(f"   Transformer: LR={transformer_lr_results['lr'][trans_best_idx]:.4f} → RMSE={transformer_lr_results['test_rmse'][trans_best_idx]:.4f}°C")

## Final Results

In [ ]:
# Create a summary table comparing the best LSTM and Transformer models (original LR=0.001)
results = pd.DataFrame({
    'Model':      ['LSTM (Bidirectional)', 'Transformer (with STL trend)'],
    'Train RMSE': [lstm_train_rmse,       trans_train_rmse],
    'Test RMSE':  [lstm_test_rmse,        trans_test_rmse],
    'Test MAE':   [lstm_test_mae,         trans_test_mae],
    'Test R²':    [lstm_test_r2,          trans_test_r2],
    'Training Time (s)': [lstm_time,           transformer_time],
}).set_index('Model').round(4)

print("Model Performance Comparison")

display(results.style
    .highlight_min(subset=['Train RMSE', 'Test RMSE', 'Test MAE', 'Training Time (s)'], color='#d4edda')
    .highlight_max(subset=['Test R²'], color='#d4edda')
    .format('{:.4f}')
    .set_caption('24-step forecast performance (ETTh1, OT column, 168h lookback)')
)

In [ ]:
# Display predictions vs actuals for both models on the test set

window_indices = [861,1723] # Select 2 test windows (indices in the test set) to visualize predictions vs actuals

fig, axes = plt.subplots(len(window_indices), 1, figsize=(14, 4 * len(window_indices)))

for ax, i in zip(axes, window_indices):
    ax.plot(lstm_test_original[i], label="Actual", color="steelblue", linewidth=2)
    ax.plot(lstm_test_pred_original[i], label="LSTM", color="orange", linestyle="--", linewidth=1.5)
    ax.plot(trans_test_pred[i], label="Transformer (STL)", color="coral", linestyle=":", linewidth=1.5)
    ax.set_title(f"24-hour Forecast Window (test sample {i})")
    ax.set_xlabel("Hour")
    ax.set_ylabel("Oil Temperature (°C)")
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()